# 3. Segment Analysis: Heterogeneous Treatment Effects

This notebook examines whether the treatment effect varies across user segments.

## Approach
- Stratify users by Feature f0 (quintiles)
- Test treatment effect **within each segment**
- Identify which segments benefit most from treatment

## Why This Matters
- Not all users respond equally to ads
- Segments with strong treatment effect = target for future campaigns
- Segments with no effect = consider alternative strategies

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

from statsmodels.stats.proportion import confint_proportions_2indep

import os
DATA_PATH = os.path.join("data", "raw", "criteo-uplift-v2.1.csv.gz")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (13979592, 16)


In [17]:
df["f0_segment"] = pd.qcut(
    df["f0"],
    q=5,
    labels=False,
    duplicates="drop"
)

df["f0_segment"] = df["f0_segment"] + 1

print(df["f0_segment"].value_counts().sort_index())

f0_segment
1    5591837
2    2795921
3    2795915
4    2795919
Name: count, dtype: int64


In [18]:
segment_counts = (
    df.groupby(["f0_segment", "treatment"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "control", 1: "treatment"})
)

segment_counts

treatment,control,treatment
f0_segment,,
1,831087,4760750
2,418979,2376942
3,422755,2373160
4,424116,2371803


In [19]:
f0_segment_metrics = (
    df.groupby(["f0_segment", "treatment"])
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

f0_segment_metrics["conversion_rate"] = (
    f0_segment_metrics["conversions"]
    / f0_segment_metrics["observations"]
)

f0_segment_results = (
    f0_segment_metrics
    .pivot(
        index="f0_segment",
        columns="treatment",
        values=["observations", "conversions", "conversion_rate"]
    )
)

f0_segment_results.columns = [
    f"{metric}_{'control' if treatment == 0 else 'treatment'}"
    for metric, treatment in f0_segment_results.columns
]

f0_segment_results = f0_segment_results.reset_index()

f0_segment_results["absolute_lift"] = (
    f0_segment_results["conversion_rate_treatment"]
    - f0_segment_results["conversion_rate_control"]
)

f0_segment_results["relative_lift"] = (
    f0_segment_results["absolute_lift"]
    / f0_segment_results["conversion_rate_control"]
)

f0_segment_results

,f0_segment,observations_control,observations_treatment,conversions_control,conversions_treatment,conversion_rate_control,conversion_rate_treatment,absolute_lift,relative_lift
0,1,831087.0,4760750.0,2989.0,28126.0,0.003596,0.005908,0.002311,0.642681
1,2,418979.0,2376942.0,662.0,5672.0,0.001580,0.002386,0.000806,0.510261
2,3,422755.0,2373160.0,258.0,1845.0,0.000610,0.000777,0.000167,0.273909
3,4,424116.0,2371803.0,154.0,1068.0,0.000363,0.000450,0.000087,0.240100


In [20]:
segment_ci_results = []

for _, row in f0_segment_results.iterrows():
    ci_low, ci_high = confint_proportions_2indep(
        count1=int(row["conversions_treatment"]),
        nobs1=int(row["observations_treatment"]),
        count2=int(row["conversions_control"]),
        nobs2=int(row["observations_control"]),
        method="wald"
    )

    segment_ci_results.append({
        "f0_segment": int(row["f0_segment"]),
        "absolute_lift": row["absolute_lift"],
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

f0_segment_ci = pd.DataFrame(segment_ci_results)

f0_segment_ci

,f0_segment,absolute_lift,ci_lower,ci_upper
0,1,0.002311,0.002165,0.002457
1,2,0.000806,0.000671,0.000942
2,3,0.000167,0.000085,0.000250
3,4,0.000087,0.000024,0.000151


In [21]:
import statsmodels.formula.api as smf

f0_model = smf.logit(
    "conversion ~ treatment * C(f0_segment)",
    data=df
).fit(disp=False)

print(f0_model.summary())

                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:             13979592
Model:                          Logit   Df Residuals:                 13979584
Method:                           MLE   Df Model:                            7
Date:                Tue, 29 Sep 2026   Pseudo R-squ.:                 0.04956
Time:                        23:54:45   Log-Likelihood:            -2.6491e+05
converged:                       True   LL-Null:                   -2.7872e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -5.6242      0.018   -306.931      0.000      -5.660      -5.588
C(f0_segment)[T.2]              -0.8245      0.043    -19.177      0.000     

In [22]:
df["f2_segment"] = pd.qcut(
    df["f2"],
    q=5,
    labels=False,
    duplicates="drop"
)

df["f2_segment"] = df["f2_segment"] + 1

print(df["f2_segment"].value_counts().sort_index())

f2_segment
1    8387755
2    2795918
3    2795919
Name: count, dtype: int64


In [23]:
f2_segment_counts = (
    df.groupby(["f2_segment", "treatment"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "control", 1: "treatment"})
)

f2_segment_counts

treatment,control,treatment
f2_segment,,
1,1255172,7132583
2,415999,2379919
3,425766,2370153


In [24]:
f2_segment_metrics = (
    df.groupby(["f2_segment", "treatment"])
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

f2_segment_metrics["conversion_rate"] = (
    f2_segment_metrics["conversions"]
    / f2_segment_metrics["observations"]
)

f2_segment_results = (
    f2_segment_metrics
    .pivot(
        index="f2_segment",
        columns="treatment",
        values=["observations", "conversions", "conversion_rate"]
    )
)

f2_segment_results.columns = [
    f"{metric}_{'control' if treatment == 0 else 'treatment'}"
    for metric, treatment in f2_segment_results.columns
]

f2_segment_results = f2_segment_results.reset_index()

f2_segment_results["absolute_lift"] = (
    f2_segment_results["conversion_rate_treatment"]
    - f2_segment_results["conversion_rate_control"]
)

f2_segment_results["relative_lift"] = (
    f2_segment_results["absolute_lift"]
    / f2_segment_results["conversion_rate_control"]
)

f2_segment_results

,f2_segment,observations_control,observations_treatment,conversions_control,conversions_treatment,conversion_rate_control,conversion_rate_treatment,absolute_lift,relative_lift
0,1,1255172.0,7132583.0,2792.0,24839.0,0.002224,0.003482,0.001258,0.565579
1,2,415999.0,2379919.0,1113.0,10507.0,0.002675,0.004415,0.001739,0.650113
2,3,425766.0,2370153.0,158.0,1365.0,0.000371,0.000576,0.000205,0.551923


In [25]:
f2_segment_ci_results = []

for _, row in f2_segment_results.iterrows():
    ci_low, ci_high = confint_proportions_2indep(
        count1=int(row["conversions_treatment"]),
        nobs1=int(row["observations_treatment"]),
        count2=int(row["conversions_control"]),
        nobs2=int(row["observations_control"]),
        method="wald"
    )

    f2_segment_ci_results.append({
        "f2_segment": int(row["f2_segment"]),
        "absolute_lift": row["absolute_lift"],
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

f2_segment_ci = pd.DataFrame(f2_segment_ci_results)

f2_segment_ci

,f2_segment,absolute_lift,ci_lower,ci_upper
0,1,0.001258,0.001165,0.001351
1,2,0.001739,0.001561,0.001918
2,3,0.000205,0.000139,0.000270


In [26]:
f2_model = smf.logit(
    "conversion ~ treatment * C(f2_segment)",
    data=df
).fit(disp=False)

print(f2_model.summary())

                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:             13979592
Model:                          Logit   Df Residuals:                 13979586
Method:                           MLE   Df Model:                            5
Date:                Tue, 29 Sep 2026   Pseudo R-squ.:                 0.01930
Time:                        23:57:51   Log-Likelihood:            -2.7335e+05
converged:                       True   LL-Null:                   -2.7872e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -6.1060      0.019   -322.280      0.000      -6.143      -6.069
C(f2_segment)[T.2]               0.1851      0.035      5.215      0.000     

In [27]:
df["f6_segment"] = pd.qcut(
    df["f6"],
    q=5,
    labels=False,
    duplicates="drop"
)

df["f6_segment"] = df["f6_segment"] + 1

print(df["f6_segment"].value_counts().sort_index())

f6_segment
1    2906669
2    3104117
3    4263261
4    3705545
Name: count, dtype: int64


In [28]:
f6_segment_counts = (
    df.groupby(["f6_segment", "treatment"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "control", 1: "treatment"})
)

f6_segment_counts

treatment,control,treatment
f6_segment,,
1,413526,2493143
2,466470,2637647
3,645157,3618104
4,571784,3133761


In [29]:
f6_segment_metrics = (
    df.groupby(["f6_segment", "treatment"])
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

f6_segment_metrics["conversion_rate"] = (
    f6_segment_metrics["conversions"]
    / f6_segment_metrics["observations"]
)

f6_segment_results = (
    f6_segment_metrics
    .pivot(
        index="f6_segment",
        columns="treatment",
        values=["observations", "conversions", "conversion_rate"]
    )
)

f6_segment_results.columns = [
    f"{metric}_{'control' if treatment == 0 else 'treatment'}"
    for metric, treatment in f6_segment_results.columns
]

f6_segment_results = f6_segment_results.reset_index()

f6_segment_results["absolute_lift"] = (
    f6_segment_results["conversion_rate_treatment"]
    - f6_segment_results["conversion_rate_control"]
)

f6_segment_results["relative_lift"] = (
    f6_segment_results["absolute_lift"]
    / f6_segment_results["conversion_rate_control"]
)

f6_segment_results

,f6_segment,observations_control,observations_treatment,conversions_control,conversions_treatment,conversion_rate_control,conversion_rate_treatment,absolute_lift,relative_lift
0,1,413526.0,2493143.0,2300.0,23440.0,0.005562,0.009402,0.003840,0.690384
1,2,466470.0,2637647.0,533.0,4646.0,0.001143,0.001761,0.000619,0.541555
2,3,645157.0,3618104.0,379.0,2766.0,0.000587,0.000764,0.000177,0.301360
3,4,571784.0,3133761.0,851.0,5859.0,0.001488,0.001870,0.000381,0.256204


In [30]:
f6_segment_ci_results = []

for _, row in f6_segment_results.iterrows():
    ci_low, ci_high = confint_proportions_2indep(
        count1=int(row["conversions_treatment"]),
        nobs1=int(row["observations_treatment"]),
        count2=int(row["conversions_control"]),
        nobs2=int(row["observations_control"]),
        method="wald"
    )

    f6_segment_ci_results.append({
        "f6_segment": int(row["f6_segment"]),
        "absolute_lift": row["absolute_lift"],
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

f6_segment_ci = pd.DataFrame(f6_segment_ci_results)

f6_segment_ci

,f6_segment,absolute_lift,ci_lower,ci_upper
0,1,0.003840,0.003583,0.004096
1,2,0.000619,0.000509,0.000728
2,3,0.000177,0.000111,0.000243
3,4,0.000381,0.000271,0.000492


In [31]:
f6_model = smf.logit(
    "conversion ~ treatment * C(f6_segment)",
    data=df
).fit(disp=False)

print(f6_model.summary())

                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:             13979592
Model:                          Logit   Df Residuals:                 13979584
Method:                           MLE   Df Model:                            7
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                 0.06694
Time:                        00:01:23   Log-Likelihood:            -2.6007e+05
converged:                       True   LL-Null:                   -2.7872e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -5.1862      0.021   -248.030      0.000      -5.227      -5.145
C(f6_segment)[T.2]              -1.5871      0.048    -32.981      0.000     

In [32]:
df["f8_segment"] = pd.qcut(
    df["f8"],
    q=5,
    labels=False,
    duplicates="drop"
)

df["f8_segment"] = df["f8_segment"] + 1

print(df["f8_segment"].value_counts().sort_index())

f8_segment
1    2848806
2    2871462
3    8259324
Name: count, dtype: int64


In [33]:
f8_segment_counts = (
    df.groupby(["f8_segment", "treatment"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "control", 1: "treatment"})
)

f8_segment_counts

treatment,control,treatment
f8_segment,,
1,412734,2436072
2,437429,2434033
3,1246774,7012550


In [34]:
f8_segment_metrics = (
    df.groupby(["f8_segment", "treatment"])
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

f8_segment_metrics["conversion_rate"] = (
    f8_segment_metrics["conversions"]
    / f8_segment_metrics["observations"]
)

f8_segment_results = (
    f8_segment_metrics
    .pivot(
        index="f8_segment",
        columns="treatment",
        values=["observations", "conversions", "conversion_rate"]
    )
)

f8_segment_results.columns = [
    f"{metric}_{'control' if treatment == 0 else 'treatment'}"
    for metric, treatment in f8_segment_results.columns
]

f8_segment_results = f8_segment_results.reset_index()

f8_segment_results["absolute_lift"] = (
    f8_segment_results["conversion_rate_treatment"]
    - f8_segment_results["conversion_rate_control"]
)

f8_segment_results["relative_lift"] = (
    f8_segment_results["absolute_lift"]
    / f8_segment_results["conversion_rate_control"]
)

f8_segment_results

,f8_segment,observations_control,observations_treatment,conversions_control,conversions_treatment,conversion_rate_control,conversion_rate_treatment,absolute_lift,relative_lift
0,1,412734.0,2436072.0,3759.0,33931.0,0.009108,0.013929,0.004821,0.529341
1,2,437429.0,2434033.0,210.0,1924.0,0.000480,0.000790,0.000310,0.646520
2,3,1246774.0,7012550.0,94.0,856.0,0.000075,0.000122,0.000047,0.619040


In [35]:
f8_segment_ci_results = []

for _, row in f8_segment_results.iterrows():
    ci_low, ci_high = confint_proportions_2indep(
        count1=int(row["conversions_treatment"]),
        nobs1=int(row["observations_treatment"]),
        count2=int(row["conversions_control"]),
        nobs2=int(row["observations_control"]),
        method="wald"
    )

    f8_segment_ci_results.append({
        "f8_segment": int(row["f8_segment"]),
        "absolute_lift": row["absolute_lift"],
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

f8_segment_ci = pd.DataFrame(f8_segment_ci_results)

f8_segment_ci

,f8_segment,absolute_lift,ci_lower,ci_upper
0,1,0.004821,0.004496,0.005146
1,2,0.000310,0.000236,0.000384
2,3,0.000047,0.000029,0.000064


In [36]:
f8_model = smf.logit(
    "conversion ~ treatment * C(f8_segment)",
    data=df
).fit(disp=False)

print(f8_model.summary())

                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:             13979592
Model:                          Logit   Df Residuals:                 13979586
Method:                           MLE   Df Model:                            5
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                  0.1850
Time:                        00:05:26   Log-Likelihood:            -2.2715e+05
converged:                       True   LL-Null:                   -2.7872e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -4.6895      0.016   -286.204      0.000      -4.722      -4.657
C(f8_segment)[T.2]              -2.9516      0.071    -41.606      0.000     

In [37]:
df["f9_segment"] = pd.qcut(
    df["f9"],
    q=5,
    labels=False,
    duplicates="drop"
)

df["f9_segment"] = df["f9_segment"] + 1

print(df["f9_segment"].value_counts().sort_index())

f9_segment
1    11568147
2     2411445
Name: count, dtype: int64


In [40]:
f9_segment_counts = (
    df.groupby(["f9_segment", "treatment"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "control", 1: "treatment"})
)

f9_segment_counts

treatment,control,treatment
f9_segment,,
1,1747126,9821021
2,349811,2061634


In [41]:
f9_segment_metrics = (
    df.groupby(["f9_segment", "treatment"])
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

f9_segment_metrics["conversion_rate"] = (
    f9_segment_metrics["conversions"] /
    f9_segment_metrics["observations"]
)

f9_segment_results = (
    f9_segment_metrics
    .pivot(
        index="f9_segment",
        columns="treatment",
        values=["observations", "conversions", "conversion_rate"]
    )
)

f9_segment_results.columns = [
    f"{metric}_{'control' if treatment == 0 else 'treatment'}"
    for metric, treatment in f9_segment_results.columns
]

f9_segment_results = f9_segment_results.reset_index()

f9_segment_results["absolute_lift"] = (
    f9_segment_results["conversion_rate_treatment"] -
    f9_segment_results["conversion_rate_control"]
)

f9_segment_results["relative_lift"] = (
    f9_segment_results["absolute_lift"] /
    f9_segment_results["conversion_rate_control"]
)

f9_segment_results

,f9_segment,observations_control,observations_treatment,conversions_control,conversions_treatment,conversion_rate_control,conversion_rate_treatment,absolute_lift,relative_lift
0,1,1747126.0,9821021.0,433.0,4090.0,0.000248,0.000416,0.000169,0.680363
1,2,349811.0,2061634.0,3630.0,32621.0,0.010377,0.015823,0.005446,0.524799


In [42]:
f9_segment_ci_results = []

for _, row in f9_segment_results.iterrows():
    ci_low, ci_high = confint_proportions_2indep(
        count1=int(row["conversions_treatment"]),
        nobs1=int(row["observations_treatment"]),
        count2=int(row["conversions_control"]),
        nobs2=int(row["observations_control"]),
        method="wald"
    )

    f9_segment_ci_results.append({
        "f9_segment": int(row["f9_segment"]),
        "absolute_lift": row["absolute_lift"],
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

f9_segment_ci = pd.DataFrame(f9_segment_ci_results)

f9_segment_ci

,f9_segment,absolute_lift,ci_lower,ci_upper
0,1,0.000169,0.000142,0.000195
1,2,0.005446,0.005069,0.005822


In [43]:
f9_model = smf.logit(
    "conversion ~ treatment * C(f9_segment)",
    data=df
).fit(disp=False)

print(f9_model.summary())

                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:             13979592
Model:                          Logit   Df Residuals:                 13979588
Method:                           MLE   Df Model:                            3
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                  0.1828
Time:                        00:09:42   Log-Likelihood:            -2.2776e+05
converged:                       True   LL-Null:                   -2.7872e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -8.3025      0.048   -172.742      0.000      -8.397      -8.208
C(f9_segment)[T.2]               3.7448      0.051     73.605      0.000     

## Segment Analysis Summary

In [2]:
segment_summary = pd.DataFrame({
    "feature": ["f0", "f2", "f6", "f8", "f9"],
    "heterogeneity_evidence": [
        "Yes",
        "No clear evidence",
        "Yes",
        "No clear evidence",
        "No clear evidence"
    ]
})

segment_summary

,feature,heterogeneity_evidence
0,f0,Yes
1,f2,No clear evidence
2,f6,Yes
3,f8,No clear evidence
4,f9,No clear evidence


### Interpretation

The segment analysis was exploratory and focused on five predefined anonymized features: f0, f2, f6, f8, and f9.

Evidence of treatment-effect heterogeneity was observed for f0 and f6, while no clear evidence was found for f2, f8, or f9 at the 5% significance level.

The feature values are anonymized, so no real-world meaning is assigned to individual features. Segment-level findings should also be interpreted cautiously because multiple comparisons can increase the chance of observing statistically significant results by chance.

These results indicate that treatment effectiveness may vary across some observed feature-defined segments, but they do not establish that any particular segment is universally "best" for treatment.

In [ ]:
# Save segment results
segment_results.to_csv('outputs/results/segment_analysis.csv', index=False)
print("✅ Segment analysis saved to outputs/results/segment_analysis.csv")